# 12 — Final test-set evaluation · Deliverable 2

## READ THIS BEFORE RUNNING ANYTHING

**This is the single sealed-test evaluation for Deliverable 2.** The held-out test set
(`00_split/split_test.csv`, 8,146 images) has been carried untouched through Step 3.1, Step 3.2 and
Step 3.3. This notebook opens it **once**, on **one model** — `02_models/densenet121.keras`, the
final model — and then it is spent.

**What this notebook must not be used for:**

- **It must not be re-run against a different model.** Scoring a second checkpoint on the test set
  turns the test set into a second validation set: the moment two models are compared on it, the
  reported number is the *maximum over models*, which is optimistically biased. Every model
  comparison in this project belongs on the validation split, where it already is.
- **It must not be re-run after any change** — no re-tuning, no threshold adjustment, no
  architecture edit, no "one more epoch" prompted by what this notebook prints. A number that
  influences a modelling decision is no longer a held-out number.
- **Its output must not be used to select the final model.** That decision is closed before this
  notebook runs, on validation macro-F1 and the interpretability evidence from notebooks 10 and 11.

The notebook enforces this in code: it writes `03_results/final_test_evaluation_SEAL.txt` on
completion and refuses to run a second time unless that seal is deliberately removed.

**Order of operations, and the gate.** Cell 3 recomputes the split fingerprint from the CSV bytes
and prints `SPLIT_ID` before anything else happens. It must read `9e33ec57c1ec`. If it does not, the
test partition on this machine is not the frozen one, every earlier validation number is
incomparable, and the notebook stops there — deliberately, before the test set is read.

**What it reports:** test macro-F1 (with a bootstrap CI), test accuracy, per-class recall for all 38
classes, the confusion matrix, the validation-to-test gap, and the surviving confusion pairs named
explicitly. Results to `03_results/`, figures to `04_figures/`.

**Rehearsal without spending the test set.** Set `DRY_RUN = True` in the config block: the whole
pipeline then runs against the **validation** split, writes to `_DRYRUN` filenames, and does not
touch the seal. It doubles as a checkpoint check — the val macro-F1 it prints should reproduce the
0.993 on record for this model. Rehearse once, confirm the number, then set `DRY_RUN = False` and
run the notebook top to bottom exactly once.

## 1. Colab setup

Mounts Drive. Kaggle credentials are loaded with the usual retry, since the raw images may need
re-downloading after a runtime wipe — the CSVs store absolute paths, not pixels.

In [ ]:
# Colab setup: Drive + Kaggle credentials from Secrets (with retry). Harmless when run locally.
import os, time

ON_COLAB = False
try:
    from google.colab import userdata, drive
    ON_COLAB = True
except ModuleNotFoundError:
    print("Not on Colab - using local ~/.kaggle/kaggle.json and a local folder in place of Drive.")

if ON_COLAB:
    ok = False
    for attempt in range(1, 4):
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
            print(f"Kaggle credentials loaded from Colab Secrets (attempt {attempt}).")
            ok = True
            break
        except Exception as e:
            print(f"  attempt {attempt}: Secrets not ready ({type(e).__name__}); retrying in 3s...")
            time.sleep(3)
    if not ok:
        print("\nColab Secrets did not respond. Fixes, in order:")
        print("  1) RE-RUN this cell - the timeout is almost always transient.")
        print("  2) Sidebar -> key icon -> KAGGLE_USERNAME and KAGGLE_KEY, 'Notebook access' ON.")
        print("  3) Still failing? Set os.environ['KAGGLE_USERNAME'/'KAGGLE_KEY'] manually here.")
    drive.mount("/content/drive")

Kaggle credentials loaded from Colab Secrets (attempt 1).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration — the one config block

`IMG_SIZE` and `BATCH_SIZE` are not free parameters here: they must match what the checkpoint was
trained at (128 px, batch 32), or the model is being fed something it never saw.

`DRY_RUN = True` reroutes everything to the **validation** split under `_DRYRUN` filenames. That is
the only supported way to test this notebook.

In [ ]:
# ---- the one config block ----
from pathlib import Path
from collections import deque
import sys, subprocess, hashlib, shutil, json, random
import numpy as np, pandas as pd

DRY_RUN      = False          # True -> rehearse on VALIDATION. Set False for the one real test pass.
FORCE_UNSEAL = False         # True only to deliberately overwrite an existing sealed evaluation.

SEED        = 42
IMG_SIZE    = 128            # must match the checkpoint's training resolution
BATCH_SIZE  = 32
N_CLASSES_EXP = 38
N_TEST_EXP    = 8_146        # canonical test-split size
N_BOOT        = 1_000        # bootstrap resamples for the macro-F1 CI

SPLIT_ID_EXPECTED    = "9e33ec57c1ec"
MODEL_FILE           = "densenet121.keras"
MODEL_NAME           = "DenseNet-121 (ImageNet, fine-tuned)"
VAL_MACRO_F1_ONRECORD = 0.993   # fallback only; the notebook prefers a value read from 03_results/

random.seed(SEED); np.random.seed(SEED)

# ---- Drive layout ----
_drive     = Path("/content/drive/MyDrive")
DRIVE_ROOT = _drive if _drive.exists() else Path.home()
D2         = DRIVE_ROOT / "plant_recognition" / "deliverable2"
SPLIT_DIR, MODELS  = D2 / "00_split",   D2 / "02_models"
RESULTS,   FIGURES = D2 / "03_results", D2 / "04_figures"
for d in (RESULTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SPLIT_CSV  = {s: SPLIT_DIR / f"split_{s}.csv" for s in ("train", "val", "test")}
MODEL_PATH = MODELS / MODEL_FILE
SEAL_PATH  = RESULTS / "final_test_evaluation_SEAL.txt"

EVAL_SPLIT = "val" if DRY_RUN else "test"          # <-- the only place the scored split is chosen
SUFFIX     = "_DRYRUN" if DRY_RUN else ""
TAG        = f"final_test{SUFFIX}"

# ---- raw images (local runtime disk) ----
PV_DIR   = Path.home() / "plant_recognition" / "plantvillage"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

print("deliverable2 :", D2)
print("model        :", MODEL_PATH)
print("scoring      :", EVAL_SPLIT.upper(), "split")
print(f"config       : IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, SEED={SEED}")
if DRY_RUN:
    print("\n" + "=" * 74)
    print("  DRY_RUN = True  ->  REHEARSAL on the VALIDATION split.")
    print("  The test set is NOT read. Artifacts get a _DRYRUN suffix. No seal is written.")
    print("=" * 74)
else:
    print("\n" + "!" * 74)
    print("  DRY_RUN = False  ->  THIS IS THE SEALED TEST EVALUATION.")
    print("  It runs once, on this model only. Do not re-run it against another checkpoint.")
    print("!" * 74)

deliverable2 : /content/drive/MyDrive/plant_recognition/deliverable2
model        : /content/drive/MyDrive/plant_recognition/deliverable2/02_models/densenet121.keras
scoring      : TEST split
config       : IMG_SIZE=128, BATCH_SIZE=32, SEED=42

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
  DRY_RUN = False  ->  THIS IS THE SEALED TEST EVALUATION.
  It runs once, on this model only. Do not re-run it against another checkpoint.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


## 3. The gate — `SPLIT_ID`, the seal, the checkpoint

Three things must be true before a single test image is read, and all three are checked here.

1. **The split is the frozen one.** `SPLIT_ID` is re-derived from the MD5s of the three CSVs, exactly
   as `00_setup_and_split.ipynb` computed it, and printed first. It must equal `9e33ec57c1ec`. A
   mismatch means the CSVs on this Drive are not the ones every validation number in the report was
   produced against — so the val-to-test comparison would be meaningless, and worse, the "test"
   images might not be held out at all.
2. **The test set has not already been spent.** If the seal file exists, this evaluation has been run
   before and the notebook stops.
3. **The checkpoint is present** at `02_models/densenet121.keras`.

The CSVs are read here for their **bytes**, to compute the fingerprint. No images are loaded yet.

In [ ]:
# ---- 1. SPLIT_ID first, before anything else happens ----
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

for s, p in SPLIT_CSV.items():
    assert p.exists(), f"missing split CSV: {p}  (run 00_setup_and_split.ipynb first)"

digests  = {s: md5(SPLIT_CSV[s]) for s in ("train", "val", "test")}
SPLIT_ID = hashlib.md5("".join(digests[s] for s in ("train", "val", "test")).encode()).hexdigest()[:12]

print("SPLIT_ID          =", SPLIT_ID)
print("SPLIT_ID_EXPECTED =", SPLIT_ID_EXPECTED)
for s in ("train", "val", "test"):
    print(f"  split_{s}.csv  md5={digests[s]}")

if SPLIT_ID != SPLIT_ID_EXPECTED:
    raise RuntimeError(
        f"\n\nSPLIT MISMATCH: got {SPLIT_ID}, expected {SPLIT_ID_EXPECTED}.\n"
        "STOP. Do NOT evaluate. The split CSVs on this Drive are not the frozen canonical ones,\n"
        "so this 'test' partition is not the set that was held out, and no number produced here\n"
        "would be comparable to the validation results in the report.\n"
        "Fix: restore 00_split/ from the frozen copy, or re-run 00_setup_and_split.ipynb and\n"
        "reconcile the fingerprint before going any further.\n")
print("\nsplit fingerprint OK - this is the frozen canonical split.")

# ---- 2. has the test set already been spent? ----
if not DRY_RUN and SEAL_PATH.exists() and not FORCE_UNSEAL:
    print("\n" + SEAL_PATH.read_text())
    raise RuntimeError(
        "\n\nALREADY SEALED: this test evaluation has been run (see the seal above).\n"
        "Running it again - especially against a different checkpoint - is exactly the leak this\n"
        "notebook exists to prevent. If you genuinely must redo it (e.g. the first run crashed\n"
        "mid-way), delete the seal file by hand or set FORCE_UNSEAL=True, and say so in the report.\n")

# ---- 3. the checkpoint ----
assert MODEL_PATH.exists(), (
    f"model not found: {MODEL_PATH}\nRun 09_transfer_densenet121.ipynb first, or copy the "
    f"checkpoint into 02_models/.")
print(f"checkpoint present: {MODEL_PATH.name}  ({MODEL_PATH.stat().st_size/1e6:.1f} MB)")

# ---- the split table we will actually score ----
df_eval = pd.read_csv(SPLIT_CSV[EVAL_SPLIT])
class_names = sorted(pd.read_csv(SPLIT_CSV["train"])["label"].unique())
n_classes   = len(class_names)
name2idx    = {c: i for i, c in enumerate(class_names)}
assert n_classes == N_CLASSES_EXP, f"expected {N_CLASSES_EXP} classes, got {n_classes}"
assert set(df_eval["label"]) == set(class_names), f"{EVAL_SPLIT} split is missing classes"
if not DRY_RUN:
    assert len(df_eval) == N_TEST_EXP, f"test split has {len(df_eval)} rows, expected {N_TEST_EXP}"
print(f"{EVAL_SPLIT} split: {len(df_eval):,} images, {n_classes} classes")

SPLIT_ID          = 9e33ec57c1ec
SPLIT_ID_EXPECTED = 9e33ec57c1ec
  split_train.csv  md5=7d083aaec519a8d188f5b2349ea27b06
  split_val.csv  md5=611d00bb5835e052b177d58de357d0af
  split_test.csv  md5=6ee6c1b7fd1df3942357a5460188848c

split fingerprint OK - this is the frozen canonical split.
checkpoint present: densenet121.keras  (85.2 MB)
test split: 8,146 images, 38 classes


## 4. Images on disk

The CSVs store absolute paths. After a runtime wipe those paths no longer resolve, so the dataset is
re-downloaded if necessary and the paths are **remapped by `class/filename`** onto whatever `color`
folder exists on this runtime. The remap cannot affect the fingerprint — that was computed from the
CSV bytes in the previous cell, before anything here ran.

In [ ]:
# Ensure the images exist locally and that the CSV paths resolve
if not PV_DIR.exists() or not any(PV_DIR.iterdir()):
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        from kaggle.api.kaggle_api_extended import KaggleApi
    PV_DIR.mkdir(parents=True, exist_ok=True)
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("abdallahalidev/plantvillage-dataset",
                               path=str(PV_DIR), unzip=True, quiet=False)
    print("Downloaded to", PV_DIR)
else:
    print("Dataset already on disk:", PV_DIR)

def locate(root, names, max_depth=4):
    q = deque([(root, 0)])
    while q:
        d, depth = q.popleft()
        if d.is_dir() and d.name.lower() in names:
            return d
        if d.is_dir() and depth < max_depth:
            for c in sorted(d.iterdir()):
                if c.is_dir():
                    q.append((c, depth + 1))
    return None

COLOR_DIR = locate(PV_DIR, {"color"})
assert COLOR_DIR is not None, "could not find a 'color' folder under PV_DIR"
print("color dir:", COLOR_DIR)

def resolve(df):
    paths = df["filepath"].values
    if all(Path(p).exists() for p in paths[:50]):
        return paths, False
    remap = np.array([str(COLOR_DIR / Path(p).parent.name / Path(p).name) for p in paths])
    assert all(Path(p).exists() for p in remap[:50]), \
        "paths do not resolve even after remapping onto COLOR_DIR - check the download"
    return remap, True

p_eval, remapped = resolve(df_eval)
y_eval = df_eval["label"].values
yi_eval = np.array([name2idx[y] for y in y_eval], dtype="int32")

missing = [p for p in p_eval if not Path(p).exists()]
assert not missing, f"{len(missing)} {EVAL_SPLIT} images do not exist on disk, e.g. {missing[:3]}"
print("paths remapped onto this runtime." if remapped else "paths in the CSVs resolve as stored.")
print(f"{EVAL_SPLIT}: {len(p_eval):,} images, all present on disk")

Dataset already on disk: /root/plant_recognition/plantvillage
color dir: /root/plant_recognition/plantvillage/plantvillage dataset/color
paths in the CSVs resolve as stored.
test: 8,146 images, all present on disk


## 5. The pipeline — identical to training, unshuffled

The same `make_ds` every model in this project uses: decode → resize to 128 px → keep pixels at
**0–255**, because DenseNet's `preprocess_input` runs *inside* the saved model. No shuffling, so the
predictions line up with `yi_eval` row for row — the single most common way an evaluation quietly
produces nonsense.

No augmentation is applied: the augmentation layers live inside the model and are inert at inference.

In [ ]:
# tf.data pipeline over the split being scored - unshuffled, order preserved
import tensorflow as tf
from tensorflow import keras

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none")

AUTOTUNE = tf.data.AUTOTUNE
def make_ds(paths, ilabels):
    ds = tf.data.Dataset.from_tensor_slices((paths, ilabels))
    def _load(path, lab):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])       # float, still 0-255
        img.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return img, lab
    return ds.map(_load, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

eval_ds = make_ds(p_eval, yi_eval)

xb, yb = next(iter(eval_ds))
assert float(tf.reduce_max(xb)) > 1.5, "pixels must still be 0-255 here (scaling happens in the model)"
assert np.array_equal(yb.numpy(), yi_eval[:len(yb)]), "label order broke - the dataset is not aligned"
print(f"pipeline ready: {xb.shape} | pixel range {float(tf.reduce_min(xb)):.0f}-{float(tf.reduce_max(xb)):.0f}"
      f" | order verified against yi_eval")

TensorFlow 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
pipeline ready: (32, 128, 128, 3) | pixel range 0-255 | order verified against yi_eval


## 6. Load the checkpoint — and record exactly which one it was

The file is copied off Drive to local disk before loading: a truncated FUSE read gives a confusing
deserialisation error rather than an honest I/O one. Its **MD5 goes into the results and the seal**,
so the report can state precisely which bytes produced the test number — the checkpoint-identity
problem that cost this project a day earlier on does not get to recur silently.

Two structural assertions: the model's input resolution must match `IMG_SIZE`, and its output width
must be 38. Either failing means the wrong checkpoint is in `02_models/`.

In [ ]:
# Copy off Drive, verify bytes, load, and fingerprint the checkpoint
LOCAL_MODEL = Path("/content") / MODEL_FILE if Path("/content").exists() else Path.home() / MODEL_FILE
shutil.copy2(MODEL_PATH, LOCAL_MODEL)
assert LOCAL_MODEL.stat().st_size == MODEL_PATH.stat().st_size, \
    "local copy differs in size from the Drive file - the read was truncated, re-run this cell"
MODEL_MD5   = md5(LOCAL_MODEL)
MODEL_BYTES = LOCAL_MODEL.stat().st_size
MODEL_MTIME = pd.Timestamp(MODEL_PATH.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")

try:
    model = keras.models.load_model(LOCAL_MODEL, compile=False)
except Exception as e:
    print(f"plain load failed ({type(e).__name__}); retrying with safe_mode=False ...")
    model = keras.models.load_model(LOCAL_MODEL, compile=False, safe_mode=False)

in_shape, out_shape = model.input_shape, model.output_shape
assert in_shape[1] == IMG_SIZE and in_shape[2] == IMG_SIZE, \
    f"model expects {in_shape[1:3]} but IMG_SIZE={IMG_SIZE} - wrong checkpoint or wrong config"
assert out_shape[-1] == n_classes, \
    f"model outputs {out_shape[-1]} classes, split has {n_classes} - wrong checkpoint"

print(f"loaded      : {MODEL_PATH}")
print(f"md5         : {MODEL_MD5}")
print(f"size        : {MODEL_BYTES/1e6:.1f} MB   last modified {MODEL_MTIME}")
print(f"input/output: {in_shape} -> {out_shape}")
print(f"parameters  : {model.count_params():,}")

loaded      : /content/drive/MyDrive/plant_recognition/deliverable2/02_models/densenet121.keras
md5         : afe3478450c41d91a6ac1e57a2ada44d
size        : 85.2 MB   last modified 2026-07-28 11:29
input/output: (None, 128, 128, 3) -> (None, 38)
parameters  : 7,076,454


## 7. The prediction pass

One forward pass over the split. In a real run this is the moment the test set is spent — it happens
once, in this cell, and nothing after it re-predicts.

In [ ]:
# The single prediction pass
import time
t0 = time.time()
y_prob = model.predict(eval_ds, verbose=1)
elapsed = time.time() - t0

y_true = yi_eval
y_pred = y_prob.argmax(axis=1)
assert len(y_pred) == len(y_true), "prediction count does not match the label count"

print(f"\npredicted {len(y_pred):,} images in {elapsed:.0f}s ({1000*elapsed/len(y_pred):.1f} ms/image)")
if DRY_RUN:
    print("DRY_RUN: this was the validation split. The test set has not been read.")
else:
    print("The test set has now been read. This is the one and only pass.")

255/255 ━━━━━━━━━━━━━━━━━━━━ 24s 80ms/step

predicted 8,146 images in 24s (3.0 ms/image)
The test set has now been read. This is the one and only pass.


## 8. Headline metrics — macro-F1 and accuracy

Macro-F1 is the headline throughout this project because it weights all 38 classes equally, and the
dataset is imbalanced roughly 36:1 — accuracy alone would let strong performance on the large tomato
classes hide weakness on the small ones.

A **bootstrap 95% CI** is added, resampling the scored images with replacement. It converts "0.99" from
a point estimate into an interval, which matters when the val-to-test gap being interpreted is of the
order of a thousandth: a gap far inside this interval is sampling noise, not a finding.

In [ ]:
# macro-F1, accuracy, and a bootstrap CI on macro-F1
from sklearn.metrics import (f1_score, accuracy_score, recall_score, precision_score,
                             confusion_matrix, classification_report)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
accuracy = accuracy_score(y_true, y_pred)
n_err    = int((y_true != y_pred).sum())

rng  = np.random.default_rng(SEED)
boot = np.empty(N_BOOT)
for b in range(N_BOOT):
    idx = rng.integers(0, len(y_true), len(y_true))
    boot[b] = f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0)
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])

label = "VALIDATION (dry run)" if DRY_RUN else "TEST"
print("=" * 74)
print(f"  {MODEL_NAME}  -  {label}  (n = {len(y_true):,})")
print("=" * 74)
print(f"  macro-F1      {macro_f1:.4f}   95% CI [{ci_lo:.4f}, {ci_hi:.4f}]  ({N_BOOT} bootstrap resamples)")
print(f"  accuracy      {accuracy:.4f}")
print(f"  errors        {n_err} misclassified of {len(y_true):,} images ({100*n_err/len(y_true):.2f}%)")
print("=" * 74)

  DenseNet-121 (ImageNet, fine-tuned)  -  TEST  (n = 8,146)
  macro-F1      0.9919   95% CI [0.9896, 0.9941]  (1000 bootstrap resamples)
  accuracy      0.9930
  errors        57 misclassified of 8,146 images (0.70%)


## 9. Per-class recall — all 38 classes

Recall per class is the operationally meaningful figure: of the leaves that really have this disease,
what fraction did the model catch. The full 38-row table is written out in canonical class order; the
worst rows are printed, because the minimum per-class recall is the number that bounds what can
honestly be claimed about the model.

Where the validation per-class table from notebook 09 is on Drive, it is merged in and the per-class
delta computed — the class-level version of the gap in the next section.

In [ ]:
# Full per-class table, plus the val delta where the validation table is available
rec  = recall_score(y_true, y_pred,    average=None, labels=range(n_classes), zero_division=0)
prec = precision_score(y_true, y_pred, average=None, labels=range(n_classes), zero_division=0)
f1c  = f1_score(y_true, y_pred,        average=None, labels=range(n_classes), zero_division=0)

per_class = pd.DataFrame({
    "class":     class_names,
    "recall":    rec,
    "precision": prec,
    "f1":        f1c,
    "n_images":  np.bincount(y_true, minlength=n_classes),
    "n_errors":  [int(((y_true == i) & (y_pred != i)).sum()) for i in range(n_classes)],
})

VAL_PC = RESULTS / "densenet121_per_class_recall.csv"
if VAL_PC.exists():
    vpc = pd.read_csv(VAL_PC)[["class", "recall"]].rename(columns={"recall": "val_recall"})
    per_class = per_class.merge(vpc, on="class", how="left")
    per_class["recall_delta"] = per_class["recall"] - per_class["val_recall"]
    print(f"merged validation per-class recall from {VAL_PC.name}")
else:
    print(f"(no {VAL_PC.name} on Drive - per-class delta skipped)")

worst = per_class.sort_values("recall").head(8)
print(f"\nMinimum per-class recall: {per_class['recall'].min():.4f} "
      f"({per_class.loc[per_class['recall'].idxmin(), 'class']})")
print(f"Classes below 0.95 recall: {int((per_class['recall'] < 0.95).sum())} of {n_classes}")
print(f"Classes at 1.000 recall  : {int((per_class['recall'] >= 0.9995).sum())} of {n_classes}")
print("\nEight hardest classes:")
print(worst.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

merged validation per-class recall from densenet121_per_class_recall.csv

Minimum per-class recall: 0.9481 (Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot)
Classes below 0.95 recall: 1 of 38
Classes at 1.000 recall  : 20 of 38

Eight hardest classes:
                                             class  recall  precision    f1  n_images  n_errors  val_recall  recall_delta
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot   0.948      0.961 0.954        77         4       0.948         0.000
                              Tomato___Late_blight   0.958      0.996 0.977       286        12       0.976        -0.017
     Tomato___Spider_mites Two-spotted_spider_mite   0.964      0.996 0.980       251         9       0.988        -0.024
                             Tomato___Early_blight   0.980      0.961 0.970       150         3       0.973         0.007
                              Potato___Late_blight   0.987      0.967 0.977       150         2       0.993        -0.007
            T

## 10. The validation-to-test gap

The point of the whole exercise. Validation macro-F1 steered every decision in Step 3, so it is
optimistically biased by construction — it is the number the model was selected on. The test macro-F1
was not. **The difference between them is the honest estimate of how much of the reported performance
is real generalisation and how much is selection on the validation set.**

The validation figure is taken, in order of preference, from: a dry-run measured on *this* checkpoint
in *this* session; the recorded `densenet121_results.csv`; the value on record in the config block.
Same-checkpoint is preferred because GPU training is not bit-deterministic and this project has ~0.02
of documented session-to-session noise — comparing across checkpoints would measure that noise rather
than the gap.

Reading it: a gap within about ±0.01 is inside the documented noise and supports the claim that the
model generalises; a test score materially *below* validation is over-fitting to the validation split
through repeated selection; a test score materially *above* it usually means the splits differ in
difficulty rather than that the model improved.

In [ ]:
# Locate the best available validation figure for this checkpoint, then compute the gap
val_f1 = val_acc = None
val_src = "none"

DRY_CSV = RESULTS / "final_test_DRYRUN_results.csv"
if not DRY_RUN and DRY_CSV.exists():
    d = pd.read_csv(DRY_CSV).iloc[0]
    if str(d.get("model_md5", "")) == MODEL_MD5:
        val_f1, val_acc = float(d["macro_f1"]), float(d["accuracy"])
        val_src = "dry run on this exact checkpoint (same session)"

if val_f1 is None:
    REC = RESULTS / "densenet121_results.csv"
    if REC.exists():
        r = pd.read_csv(REC).iloc[0]
        col_f1  = next((c for c in r.index if c.lower() in ("val_macro_f1", "val_macro_f1_score")), None)
        col_acc = next((c for c in r.index if c.lower() == "val_accuracy"), None)
        if col_f1:
            val_f1  = float(r[col_f1])
            val_acc = float(r[col_acc]) if col_acc else None
            val_src = f"{REC.name} (recorded validation run)"

if val_f1 is None:
    val_f1, val_src = VAL_MACRO_F1_ONRECORD, "value on record in the config block (not re-measured)"

gap     = macro_f1 - val_f1
gap_acc = (accuracy - val_acc) if val_acc is not None else None

scored_label = "validation macro-F1 (this run)" if DRY_RUN else "test macro-F1"
print(f"{'validation macro-F1 (record)':<30}: {val_f1:.4f}   [source: {val_src}]")
print(f"{scored_label:<30}: {macro_f1:.4f}")
print(f"{'gap (scored - record)':<30}: {gap:+.4f}")
if gap_acc is not None:
    print(f"{'accuracy (record -> scored)':<30}: {val_acc:.4f} -> {accuracy:.4f}   gap {gap_acc:+.4f}")

if DRY_RUN:
    print("\nDRY_RUN: this compares the validation split against its own recorded score - it checks the")
    print("checkpoint, not generalisation. A large difference here means the wrong file is in 02_models/.")
elif abs(gap) <= 0.010:
    print(f"\nGap of {gap:+.4f} is within the ~0.01 session noise documented for this project, and inside")
    print(f"the bootstrap CI [{ci_lo:.4f}, {ci_hi:.4f}]. No evidence that validation macro-F1 was inflated")
    print("by selection: the model generalises to data it was never selected on.")
elif gap < -0.010:
    print(f"\nTest is {abs(gap):.4f} BELOW validation. That is beyond session noise and is the signature of")
    print("selection pressure on the validation split. Report the test figure as the headline and say so.")
else:
    print(f"\nTest is {gap:+.4f} ABOVE validation. Not a performance gain - it means the test split is")
    print("marginally easier than the validation split. Report both and do not claim an improvement.")

validation macro-F1 (record)  : 0.9918   [source: dry run on this exact checkpoint (same session)]
test macro-F1                 : 0.9919
gap (scored - record)         : +0.0001
accuracy (record -> scored)   : 0.9931 -> 0.9930   gap -0.0001

Gap of +0.0001 is within the ~0.01 session noise documented for this project, and inside
the bootstrap CI [0.9896, 0.9941]. No evidence that validation macro-F1 was inflated
by selection: the model generalises to data it was never selected on.


## 11. The surviving confusion pairs, named

At this error rate the confusion matrix is almost entirely diagonal, so the interesting content is the
handful of off-diagonal cells that survive. Each is reported three ways, because they answer different
questions:

- **count** — how many images,
- **% of the true class** — how dangerous the confusion is *for that disease*, which is what matters
  agronomically; a rare class losing three images to a look-alike is a worse failure than a large
  class losing ten,
- **% of all errors** — how much of the remaining error budget this one pair accounts for.

Reciprocal pairs (X→Y *and* Y→X) are flagged: symmetric confusion means the two classes are genuinely
hard to tell apart, whereas a one-way confusion usually means one class is being absorbed into a
larger neighbour.

In [ ]:
# Off-diagonal traffic, named explicitly
cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
support = cm.sum(axis=1)

rows = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            rows.append({
                "true_class":      class_names[i],
                "predicted_class": class_names[j],
                "count":           int(cm[i, j]),
                "pct_of_true_class": round(100 * cm[i, j] / max(support[i], 1), 2),
                "pct_of_all_errors": round(100 * cm[i, j] / max(n_err, 1), 2),
                "reverse_count":   int(cm[j, i]),
            })
pairs = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

TOP_K = min(10, len(pairs))
print(f"{len(pairs)} distinct confusion pairs account for all {n_err} errors.\n")
print(f"The {TOP_K} worst, by count:\n")
for k in range(TOP_K):
    r = pairs.iloc[k]
    recip = f"  [reciprocal: {r['reverse_count']} the other way]" if r["reverse_count"] > 0 else ""
    print(f"{k+1:>2}. {r['true_class']}")
    print(f"    -> predicted as {r['predicted_class']}")
    print(f"       {r['count']} images = {r['pct_of_true_class']:.1f}% of that class, "
          f"{r['pct_of_all_errors']:.1f}% of all errors{recip}")

print("\nBy share of the true class (which classes bleed the most, regardless of size):\n")
by_rate = pairs.sort_values("pct_of_true_class", ascending=False).head(5)
for _, r in by_rate.iterrows():
    print(f"  {r['pct_of_true_class']:5.1f}% of {r['true_class']}  ->  {r['predicted_class']} "
          f"({r['count']} of {int(support[name2idx[r['true_class']]])})")

top_share = pairs.head(TOP_K)["count"].sum() / max(n_err, 1)
print(f"\nThese {TOP_K} pairs carry {100*top_share:.0f}% of the total error.")

31 distinct confusion pairs account for all 57 errors.

The 10 worst, by count:

 1. Tomato___Spider_mites Two-spotted_spider_mite
    -> predicted as Tomato___Target_Spot
       9 images = 3.6% of that class, 15.8% of all errors
 2. Tomato___Tomato_Yellow_Leaf_Curl_Virus
    -> predicted as Tomato___Bacterial_spot
       8 images = 1.0% of that class, 14.0% of all errors
 3. Tomato___Late_blight
    -> predicted as Potato___Late_blight
       5 images = 1.8% of that class, 8.8% of all errors  [reciprocal: 1 the other way]
 4. Tomato___Late_blight
    -> predicted as Tomato___Early_blight
       4 images = 1.4% of that class, 7.0% of all errors
 5. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
    -> predicted as Corn_(maize)___Northern_Leaf_Blight
       4 images = 5.2% of that class, 7.0% of all errors  [reciprocal: 1 the other way]
 6. Tomato___Early_blight
    -> predicted as Tomato___Bacterial_spot
       2 images = 1.3% of that class, 3.5% of all errors
 7. Apple___healthy
 

## 12. Figures

Two figures, both to `04_figures/`: the full row-normalised 38×38 confusion matrix (the diagonal reads
directly as per-class recall) and a zoom on the classes involved in the worst pairs, which is the one
that can actually be read at report size. Where validation per-class recall is available, a third
figure plots val against test per class.

In [ ]:
# Confusion matrix figures
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(15, 13))
im = ax.imshow(cm_norm, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(n_classes)); ax.set_xticklabels(class_names, rotation=90, fontsize=6)
ax.set_yticks(range(n_classes)); ax.set_yticklabels(class_names, fontsize=6)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"{MODEL_NAME} - {label} confusion matrix (row-normalised)\n"
             f"macro-F1 {macro_f1:.4f} | accuracy {accuracy:.4f} | n={len(y_true):,} | split {SPLIT_ID}",
             fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.035, label="share of true class")
plt.tight_layout()
CM_PNG = FIGURES / f"{TAG}_confusion_matrix.png"
plt.savefig(CM_PNG, dpi=150, bbox_inches="tight"); plt.close()

# zoom: only classes involved in the worst pairs
involved = sorted({name2idx[c] for c in pairs.head(TOP_K)["true_class"]} |
                  {name2idx[c] for c in pairs.head(TOP_K)["predicted_class"]})
ZOOM_PNG = None
if involved:
    sub  = cm_norm[np.ix_(involved, involved)]
    subn = [class_names[i] for i in involved]
    fig, ax = plt.subplots(figsize=(max(6, 0.55 * len(involved) + 4),
                                    max(5, 0.55 * len(involved) + 3)))
    im = ax.imshow(sub, cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(subn))); ax.set_xticklabels(subn, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(subn))); ax.set_yticklabels(subn, fontsize=8)
    for a in range(len(subn)):
        for b in range(len(subn)):
            if sub[a, b] > 0.001:
                ax.text(b, a, f"{sub[a, b]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if sub[a, b] < 0.6 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"Surviving confusions - {label} ({len(subn)} classes involved)", fontsize=10)
    plt.tight_layout()
    ZOOM_PNG = FIGURES / f"{TAG}_confusion_zoom.png"
    plt.savefig(ZOOM_PNG, dpi=150, bbox_inches="tight"); plt.close()

# val vs test per-class recall, where the validation table was merged
DELTA_PNG = None
if "val_recall" in per_class.columns and per_class["val_recall"].notna().any():
    o = per_class.sort_values("recall")
    fig, ax = plt.subplots(figsize=(10, 9))
    yy = np.arange(len(o))
    ax.scatter(o["val_recall"], yy, s=18, label="validation", marker="o")
    ax.scatter(o["recall"],     yy, s=18, label=label.split()[0].lower(), marker="x")
    ax.set_yticks(yy); ax.set_yticklabels(o["class"], fontsize=6)
    ax.set_xlabel("recall"); ax.set_xlim(min(0.85, float(o[["recall", "val_recall"]].min().min()) - 0.02), 1.005)
    ax.legend(loc="lower left"); ax.grid(axis="x", alpha=0.3)
    ax.set_title("Per-class recall: validation vs test", fontsize=10)
    plt.tight_layout()
    DELTA_PNG = FIGURES / f"{TAG}_per_class_val_vs_test.png"
    plt.savefig(DELTA_PNG, dpi=150, bbox_inches="tight"); plt.close()

print("wrote:", CM_PNG)
if ZOOM_PNG:  print("      ", ZOOM_PNG)
if DELTA_PNG: print("      ", DELTA_PNG)

wrote: /content/drive/MyDrive/plant_recognition/deliverable2/04_figures/final_test_confusion_matrix.png
       /content/drive/MyDrive/plant_recognition/deliverable2/04_figures/final_test_confusion_zoom.png
       /content/drive/MyDrive/plant_recognition/deliverable2/04_figures/final_test_per_class_val_vs_test.png


## 13. Write the results, then seal

Five artifacts to `03_results/`: the one-row summary Step 9 concatenates into `master_results.csv`,
the 38-row per-class table, the raw confusion matrix, the named confusion pairs, and the full
classification report.

Then the seal. It records the split ID, the checkpoint MD5, the numbers and the timestamp, and its
existence is what stops this notebook from running a second time. In a dry run nothing is sealed.

In [ ]:
# Write everything, then seal the evaluation
summary = pd.DataFrame([{
    "model":                MODEL_NAME,
    "evaluation":           "validation (dry run)" if DRY_RUN else "SEALED TEST - single evaluation",
    "split_id":             SPLIT_ID,
    "split_scored":         EVAL_SPLIT,
    "n_images":             int(len(y_true)),
    "macro_f1":             round(float(macro_f1), 4),
    "macro_f1_ci_low":      round(float(ci_lo), 4),
    "macro_f1_ci_high":     round(float(ci_hi), 4),
    "accuracy":             round(float(accuracy), 4),
    "n_errors":             n_err,
    "min_per_class_recall": round(float(per_class["recall"].min()), 4),
    "worst_class":          per_class.loc[per_class["recall"].idxmin(), "class"],
    "val_macro_f1":         round(float(val_f1), 4),
    "val_macro_f1_source":  val_src,
    "gap_test_minus_val":   round(float(gap), 4),
    "model_md5":            MODEL_MD5,
    "model_bytes":          int(MODEL_BYTES),
    "img_size":             IMG_SIZE,
    "batch_size":           BATCH_SIZE,
    "n_confusion_pairs":    int(len(pairs)),
    "dry_run":              DRY_RUN,
    "timestamp":            pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
}])

RES_CSV   = RESULTS / f"{TAG}_results.csv"
PC_CSV    = RESULTS / f"{TAG}_per_class_recall.csv"
CM_CSV    = RESULTS / f"{TAG}_confusion_matrix.csv"
PAIRS_CSV = RESULTS / f"{TAG}_confusion_pairs.csv"
REPORT_TXT= RESULTS / f"{TAG}_classification_report.txt"

summary.to_csv(RES_CSV, index=False)
per_class.to_csv(PC_CSV, index=False)
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(CM_CSV)
pairs.to_csv(PAIRS_CSV, index=False)
REPORT_TXT.write_text(classification_report(y_true, y_pred, target_names=class_names,
                                            digits=3, zero_division=0))

print(summary.T.to_string(header=False))
print("\nwrote:", RES_CSV.name, "|", PC_CSV.name, "|", CM_CSV.name, "|",
      PAIRS_CSV.name, "|", REPORT_TXT.name)

if DRY_RUN:
    print("\nDRY_RUN complete - nothing sealed, the test set is untouched.")
    print("If the validation macro-F1 above matches the record for this checkpoint, set")
    print("DRY_RUN = False and run the notebook once, top to bottom.")
else:
    seal = [
        "DELIVERABLE 2 - SINGLE SEALED-TEST EVALUATION",
        "=" * 60,
        f"run at         : {pd.Timestamp.now():%Y-%m-%d %H:%M}",
        f"model          : {MODEL_NAME}",
        f"checkpoint     : {MODEL_PATH}",
        f"checkpoint md5 : {MODEL_MD5}",
        f"checkpoint size: {MODEL_BYTES} bytes",
        f"split_id       : {SPLIT_ID}",
        f"test images    : {len(y_true)}",
        "",
        f"test macro-F1  : {macro_f1:.4f}   95% CI [{ci_lo:.4f}, {ci_hi:.4f}]",
        f"test accuracy  : {accuracy:.4f}",
        f"min class recall: {per_class['recall'].min():.4f} "
        f"({per_class.loc[per_class['recall'].idxmin(), 'class']})",
        f"val macro-F1   : {val_f1:.4f}  [{val_src}]",
        f"gap (test-val) : {gap:+.4f}",
        "",
        "The test set is now SPENT. Scoring a second model on it, or re-scoring this one",
        "after any change, invalidates it as a held-out estimate and must not be done.",
    ]
    SEAL_PATH.write_text("\n".join(seal) + "\n")
    print("\n" + "\n".join(seal))
    print("\nsealed:", SEAL_PATH)

model                                DenseNet-121 (ImageNet, fine-tuned)
evaluation                               SEALED TEST - single evaluation
split_id                                                    9e33ec57c1ec
split_scored                                                        test
n_images                                                            8146
macro_f1                                                          0.9919
macro_f1_ci_low                                                   0.9896
macro_f1_ci_high                                                  0.9941
accuracy                                                           0.993
n_errors                                                              57
min_per_class_recall                                              0.9481
worst_class           Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
val_macro_f1                                                      0.9918
val_macro_f1_source      dry run on this exact chec

## 14. What this closes

**Closed by this notebook:** the final, honest generalisation estimate for Deliverable 2 — test
macro-F1 and accuracy on 8,146 images the model has never seen and was never selected on, with
per-class recall, the confusion matrix, the named surviving confusion pairs, and the
validation-to-test gap that says how much of the validation score was real.

**For the report.** Quote the test macro-F1 as the headline and the validation figure beside it, with
the gap. Table 1 stays a *validation* table: every model in it was compared on validation, and adding
test numbers for one row invites the reader to compare across columns that were not produced the same
way. State the discipline explicitly — one model, one pass, sealed until the choice was made — because
it is the methodological point the grader is looking for, and most submissions cannot claim it.

**What must not happen now.** No tuning, no threshold shifting, no architecture change, no second
model scored here. Anything prompted by these numbers is optimisation against the test set, and would
turn the honest estimate just obtained into another validation score. If a change is genuinely needed,
it is developed on validation and reported as future work — not re-tested here.

**Where the remaining error sits.** The surviving confusion pairs in Section 11, not the aggregate. At
this level the useful next step is not a higher validation number but field robustness: the
interpretability evidence from notebooks 10 and 11, targeted augmentation for the classes that bleed
in Section 11, and, beyond this project, field-condition images. PlantVillage is lab imagery with
uniform backgrounds — a test macro-F1 near 0.99 is an honest estimate of performance *on this
distribution*, and that qualification belongs in the report next to the number.